
# Deepchecks (Tabular) – Ejemplo de *Train/Test Validation* en Google Colab

Este notebook muestra cómo ejecutar **checks relevantes** de Deepchecks para comparar **train vs test**:

- **Train Test Drift (multivariante)** → `MultivariateDrift`
- **Train Test Feature Drift (por columna)** → `FeatureDrift` (o `TrainTestFeatureDrift` según versión)
- **New Label / New Categories** → `NewLabelTrainTest` y `CategoryMismatchTrainTest`
- **Train Test Label Drift** → `LabelDrift`
- **Train Test Leakage (solapamiento de IDs)** → `IndexTrainTestLeakage`
- **Train Test Feature Correlation (cambio relación feature-label)** → `FeatureLabelCorrelationChange`





## Descripción breve de los checks

### `MultivariateDrift`
Detecta si la **distribución conjunta** de varias variables ha cambiado entre el conjunto de referencia y el conjunto actual. Es útil cuando el problema no está en una sola variable, sino en la **combinación de varias features**.

### `FeatureDrift`
Evalúa si una o varias **variables de entrada** han cambiado su distribución con respecto al dataset de referencia. Sirve para identificar **data drift** en columnas concretas, como cambios en medias, rangos o frecuencias.

### `LabelDrift`
Comprueba si la **variable objetivo** (`label` o `target`) ha cambiado su distribución entre dos conjuntos de datos. Ayuda a detectar si el comportamiento de la clase o del valor a predecir ya no es el mismo que durante el entrenamiento.

### `NewLabelTrainTest`
Detecta si en el conjunto de test aparecen **etiquetas nuevas** que no existían en el conjunto de entrenamiento. Esto puede causar errores o degradar el rendimiento del modelo, especialmente en problemas de clasificación.

### `CategoryMismatchTrainTest`
Verifica si existen **categorías diferentes** entre train y test en variables categóricas. Por ejemplo, si en test aparece un valor categórico no visto antes, o si faltan categorías presentes en entrenamiento.

### `IndexTrainTestLeakage`
Comprueba si hay posible **fuga de datos** entre train y test a través del índice. Esto ocurre cuando ambos conjuntos comparten registros o identificadores, lo que puede inflar artificialmente el rendimiento del modelo.

### `FeatureLabelCorrelationChange`
Mide si ha cambiado la **relación entre las variables de entrada y la etiqueta objetivo** entre dos datasets. Es útil para detectar situaciones en las que las features siguen existiendo, pero ya no explican el target de la misma forma.

In [1]:
!pip install "deepchecks" -q


import numpy as np
np.Inf = np.inf
np.NINF = -np.inf
np.PINF = np.inf
np.NaN = np.nan



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:

# 2) Imports
import numpy as np
import pandas as pd

from deepchecks.tabular import Dataset

# Imports con fallback para compatibilidad entre versiones
try:
    from deepchecks.tabular.checks.train_test_validation import (
        MultivariateDrift,
        FeatureDrift,
        LabelDrift,
        NewLabelTrainTest,
        CategoryMismatchTrainTest,
        IndexTrainTestLeakage,
        FeatureLabelCorrelationChange,
    )
except Exception:
    # Algunas versiones exponen checks desde deepchecks.tabular.checks
    from deepchecks.tabular.checks import (
        MultivariateDrift,
        FeatureDrift,
        LabelDrift,
        NewLabelTrainTest,
        CategoryMismatchTrainTest,
        IndexTrainTestLeakage,
        FeatureLabelCorrelationChange,
    )

np.random.seed(42)


c:\Users\tteru\.pyenv\pyenv-win\versions\3.12.8\Lib\site-packages\deepchecks\core\serialization\dataframe\html.py:16: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


In [3]:
# 3) Carga de un dataset real (Titanic) y división en Train/Test
from sklearn.model_selection import train_test_split

# Cargamos un dataset público real
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

# Pequeña limpieza: quitamos nulos en columnas clave y descartamos texto libre (Name, Ticket, Cabin)
df = df.dropna(subset=['Age', 'Embarked']).reset_index(drop=True)
df = df.drop(columns=["Name", "Ticket", "Cabin"])

# Dividimos en Train (70%) y Test (30%)
train, test = train_test_split(df, test_size=0.3, random_state=42)

print("Train shape:", train.shape)
print("Test shape:", test.shape)

Train shape: (498, 9)
Test shape: (214, 9)


In [5]:
# 4) Crear objetos Dataset de Deepchecks adaptados al nuevo dataset (Titanic)

# ¡Importante! Aquí cambiamos "label" por "Survived"
label_col = "Survived"  
index_col = "PassengerId"
cat_features = ["Pclass", "Sex", "Embarked"]

# Creamos los objetos Dataset
train_ds = Dataset(train, label=label_col, cat_features=cat_features, index_name=index_col)
test_ds  = Dataset(test,  label=label_col, cat_features=cat_features, index_name=index_col)

train_ds, test_ds

(--------- Dataset Description ----------
 
                          Column           DType                 Kind Additional Info
 0                   PassengerId         integer                Index                
 1                      Survived         integer                                     
 2                        Pclass         integer  Categorical Feature                
 3                           Sex          string  Categorical Feature                
 4                           Age        floating    Numerical Feature                
 5                         SibSp         integer    Numerical Feature                
 6                         Parch         integer    Numerical Feature                
 7                          Fare        floating    Numerical Feature                
 8                      Embarked          string  Categorical Feature                
 
 
 ----------- Dataset Content ------------
 
                     PassengerId        Survived

## 5) Ejecutar checks (uno a uno)

> En Colab/Jupyter, `result.show()` renderiza un reporte interactivo.


In [6]:
# 4) Crear objetos Dataset de Deepchecks adaptados al nuevo dataset

# Definimos el target, el índice y las variables categóricas del Titanic
label_col = "Survived"  # Este es el Target
index_col = "PassengerId"
cat_features = ["Pclass", "Sex", "Embarked"]

# Creamos los objetos Dataset
train_ds = Dataset(train, label=label_col, cat_features=cat_features, index_name=index_col)
test_ds  = Dataset(test,  label=label_col, cat_features=cat_features, index_name=index_col)

train_ds, test_ds

(--------- Dataset Description ----------
 
                          Column           DType                 Kind Additional Info
 0                   PassengerId         integer                Index                
 1                      Survived         integer                                     
 2                        Pclass         integer  Categorical Feature                
 3                           Sex          string  Categorical Feature                
 4                           Age        floating    Numerical Feature                
 5                         SibSp         integer    Numerical Feature                
 6                         Parch         integer    Numerical Feature                
 7                          Fare        floating    Numerical Feature                
 8                      Embarked          string  Categorical Feature                
 
 
 ----------- Dataset Content ------------
 
                     PassengerId        Survived

In [7]:

# 5.2 Train Test Feature Drift – drift por columna (numéricas + categóricas)
res_feat_drift = FeatureDrift().run(train_ds, test_ds)
res_feat_drift.show()


In [8]:

# 5.3 New Categories – categorías nuevas en test (p.ej. 'Sevilla')
res_new_cat = CategoryMismatchTrainTest().run(train_ds, test_ds)
res_new_cat.show()


In [9]:

# 5.4 New Labels – labels nuevas en test (p.ej. 'UNKNOWN')
res_new_label = NewLabelTrainTest().run(train_ds, test_ds)
res_new_label.show()


In [10]:

# 5.5 Label Drift – cambio en distribución de la etiqueta
res_label_drift = LabelDrift().run(train_ds, test_ds)
res_label_drift.show()


In [11]:

# 5.6 Train Test Leakage – solapamiento de IDs (index leakage)
res_index_leak = IndexTrainTestLeakage().run(train_ds, test_ds)
res_index_leak.show()


In [12]:

# 5.7 Feature Correlation (feature-label) – cambio en relación con la etiqueta / leakage
res_flc = FeatureLabelCorrelationChange().run(train_ds, test_ds)
res_flc.show()


## 6) (Opcional) Exportar a HTML

Esto es útil para adjuntar el reporte a documentación (Confluence/SharePoint) o enviarlo por correo.


In [15]:
# Guardar el reporte en HTML localmente
output_file = "deepchecks_resultado_real.html"

# Aquí guardamos el resultado del check (en este caso, res_feat_drift de la celda 6)
# Si prefieres guardar la suite completa u otro check, pon su variable aquí
res_feat_drift.save_as_html(output_file)

print(f"✅ Reporte guardado con éxito como: {output_file}")
print("Revisa el explorador de archivos de la izquierda en Visual Studio. El HTML estará en la misma carpeta que tu notebook.")

✅ Reporte guardado con éxito como: deepchecks_resultado_real.html
Revisa el explorador de archivos de la izquierda en Visual Studio. El HTML estará en la misma carpeta que tu notebook.
